# Explore a disposable source project

Setup creates all example files in a temporary directory. Direct Python cells search and edit those saved files without an AI connection. Then connect ChatGPT or an API provider and ask the final question to locate the `visits` function and explain its saved source. Searches use the selected kernel machine; an explicit temporary path keeps this example independent of the kernel's working directory.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from nbinlineai.tools import (
    document_outline, file_replace_checked,
    file_strs_replace, notebook_outline, read_document_section,
    search_files, search_notebooks, source_doc, tool_catalog, tools_markdown,
    view_file_hashes,
)
import hashlib
import json

scratch = TemporaryDirectory(prefix="nbinlineai-project-")
project = Path(scratch.name)
source_file = project / "observations.py"
source_file.write_text('"""Flower observations."""\n\ndef visits(count: int = 4) -> int:\n    """Return twice the observation count."""\n    return count * 2\n', encoding="utf-8")
notes_file = project / "notes.md"
notes_file.write_text("# Field notes\nA small source project.\n## Counts\nSee [Python docs](https://docs.python.org/3/).\n", encoding="utf-8")
notebook_file = project / "saved.ipynb"
notebook_file.write_text(json.dumps({"cells": [{"cell_type": "markdown", "id": "stable-note", "source": ["# Notebook note\n", "Count visits here."]}], "metadata": {}, "nbformat": 4, "nbformat_minor": 5}), encoding="utf-8")
print("Disposable project ready")
print(tool_catalog("code"))

Disposable project ready
code (9 tools): search_files, search_notebooks, source_doc, source_files, view_file, view_file_hashes, file_replace_checked, inspect_python, show_doc


## Search saved project files

The search tools inspect files on disk. `search_notebooks` reports saved cell IDs; it cannot see unsaved edits in this open notebook.

In [2]:
print(search_files("Return twice", path=str(project), pattern="*.py"))
print(search_notebooks("Count visits", path=str(project)))
print(notebook_outline(str(notebook_file)))

Saved-file matches under [temporary local path]
[temporary local path] """Return twice the observation count."""
Saved notebook matches under [temporary local path]
[temporary local path] Count visits here.
[temporary local path] 1 cells
0: markdown id=stable-note # Notebook note Count visits here.


## Read static Python source

`source_doc` parses a saved `.py` file without importing or running it. Its source signature may differ from a live decorated function. `document_outline` can also list Python definitions, with copied addresses that become stale if the file changes.


In [3]:
print(source_doc(str(source_file), "visits"))
print(document_outline(str(source_file)))


[temporary local path] visits
def visits(count: int = 4) -> int:
Return twice the observation count.
Outline: [temporary local path]
s:c371aac23f5c126bf152069f7795ef79cf60c09614b723b38a5866f16ceb763a:0 Entire document (lines 1-5)
s:c371aac23f5c126bf152069f7795ef79cf60c09614b723b38a5866f16ceb763a:1   def visits (lines 3-5)


## Read a section and make checked edits

The Markdown outline gives opaque section addresses. Any change to the file makes a copied address stale; request a fresh outline after editing. The file hash covers the entire saved file, so an edit made after the view will cause `file_replace_checked` to refuse a stale digest.


In [4]:
outline = document_outline(str(notes_file))
print(outline)
counts_address = next(line.split()[0] for line in outline.splitlines() if "Counts" in line)
print(read_document_section(str(notes_file), counts_address))
print(view_file_hashes(str(notes_file), start_line=1, end_line=4))
expected_digest = hashlib.sha256(notes_file.read_bytes()).hexdigest()
print(file_replace_checked(str(notes_file), "A small source project.", "A checked source project.", expected_digest))
print(file_strs_replace(str(notes_file), ["Field notes", "Counts"], ["Study notes", "Totals"]))
print(notes_file.read_text())

Outline: [temporary local path]
s:7155940886116553611fb4f80bacbac56baba4a3b02887358802bdf1d9ecf7e2:0 Entire document (lines 1-4)
s:7155940886116553611fb4f80bacbac56baba4a3b02887358802bdf1d9ecf7e2:1   Field notes (lines 1-4)
s:7155940886116553611fb4f80bacbac56baba4a3b02887358802bdf1d9ecf7e2:2     Counts (lines 3-4)
[temporary local path] s:7155940886116553611fb4f80bacbac56baba4a3b02887358802bdf1d9ecf7e2:2
## Counts
See [Python docs](https://docs.python.org/3/).

[temporary local path]
sha256: 7155940886116553611fb4f80bacbac56baba4a3b02887358802bdf1d9ecf7e2
1: # Field notes
2: A small source project.
3: ## Counts
4: See [Python docs](https://docs.python.org/3/).
Updated: [temporary local path]
@@ -1,3 +1,3 @@
 # Field notes
-A small source project.
+A checked source project.
 ## Counts
Updated: [temporary local path]
@@ -1,4 +1,4 @@
-# Field notes
+# Study notes
 A checked source project.
-## Counts
+## Totals
 See [Python docs](https://docs.python.org/3/).
# Study notes
A checked source

## Ask AI about the same project

Run setup first. These declarations offer just the four tools needed to explore the disposable project. Keep this note's **Tools** checkbox on while running the next question.

- &`search_files`
- &`source_doc`
- &`document_outline`
- &`read_document_section`


In the disposable project at $`project`, use the offered tools to find the `visits` definition and explain its source signature. Read the saved Markdown outline and its Counts/Totals section. Do not change files.

`visits` is defined in `observations.py` as `def visits(count: int = 4) -> int:`. It takes an integer count (default `4`) and returns twice that count. The saved Markdown outline lists **Study notes → Totals**; the Totals section links to the Python documentation.

**Observed live tool calls for the answer above** (actual subscription events; private fields shortened).

| Tool | Result state | Submitted arguments | Observed result |
| --- | --- | --- | --- |
| `search_files` | completed | {"query":"def visits","path":"[local path]","pattern":"*.py"} | 'Saved-file matches under [local path] def visits(count: int = 4) -> int:' |
| `source_doc` | completed | {"path":"[local path]","symbol":"visits"} | '[local path] visits\ndef visits(count: int = 4) -> int:\nReturn twice the observation count.' |
| `document_outline` | completed | {"path":"[local path]"} | 'Outline: [local path] Entire document (lines 1-4)\ns:1ea690bdc454b3bc5bd0ac36b185cfc6a021990af4b8d19454b69ab36d33ac28:1   Study notes (lines 1-4)\ns:1ea690bdc454b3bc5bd0ac36b185cfc6a021990af4b8d19454b69ab36d33ac28:2     Totals (lines 3-4)' |
| `read_document_section` | completed | {"path":"[local path]","section":"s:1ea690bdc454b3bc5bd0ac36b185cfc6a021990af4b8d19454b69ab36d33ac28:2"} | '[local path] s:1ea690bdc454b3bc5bd0ac36b185cfc6a021990af4b8d19454b69ab36d33ac28:2\n## Totals\nSee [Python docs](https://docs.python.org/3/).\n' |
